# 1D-CNN for Human Activity Recognition

This notebook implements and evaluates a 1D Convolutional Neural Network (1D-CNN)
for Human Activity Recognition using the UCI Human Activity Recognition Using
Smartphones dataset.

The experiment follows the group's common preprocessing pipeline and uses the
preprocessed dataset stored in `processed_Data/har_processed.npz`.

The 1D-CNN experiment consists of multiple candidate CNN configurations.
Candidate models are trained using the training set and compared using the
validation set. The test set is kept completely unseen during model selection
and is used only for the final evaluation of the selected model.

The experiment evaluates predictive performance, generalization, model
complexity, training efficiency, inference efficiency, and training behaviour.

## 1. Imports and Reproducibility

The experiment uses fixed random seeds to improve reproducibility across model
training runs.

The random seed is fixed before loading and training models. No test-set
information is used during model development or candidate selection.

In [1]:
import os
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Reproducibility
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)
print("Random seed:", SEED)

TensorFlow version: 2.21.0
Random seed: 42


## 2. Load Preprocessed Data

The common preprocessing pipeline has already performed the required preprocessing
and generated `har_processed.npz`.

The CNN uses this processed data directly. No additional normalization or
data splitting is performed in this notebook.

The dataset contains separate training, validation, and test sets:

- Training set: used to train candidate models
- Validation set: used for candidate comparison and model selection
- Test set: reserved exclusively for final evaluation

In [2]:
# Load the shared preprocessed dataset
data_path = "../processed_Data/har_processed.npz"

data = np.load(data_path)

# Input data
X_train = data["X_train"]
X_val = data["X_val"]
X_test = data["X_test"]

# Labels
y_train = data["y_train"]
y_val = data["y_val"]
y_test = data["y_test"]

# Metadata
activity_names = data["activity_names"]
signal_names = data["signal_names"]

print("Data loaded successfully.")
print()
print("Training data   :", X_train.shape)
print("Validation data :", X_val.shape)
print("Test data       :", X_test.shape)
print()
print("Activities:", activity_names)
print("Signals:", signal_names)

Data loaded successfully.

Training data   : (5551, 128, 9)
Validation data : (1801, 128, 9)
Test data       : (2947, 128, 9)

Activities: ['Walking' 'Walking Upstairs' 'Walking Downstairs' 'Sitting' 'Standing'
 'Laying']
Signals: ['body_acc_x' 'body_acc_y' 'body_acc_z' 'body_gyro_x' 'body_gyro_y'
 'body_gyro_z' 'total_acc_x' 'total_acc_y' 'total_acc_z']


## 3. Verify Dataset Structure

The 1D-CNN expects each sample to preserve its temporal structure.

Each sample contains:

- 128 time steps
- 9 sensor signals

Therefore, the CNN input shape is `(128, 9)`.

The problem is a six-class classification task.

In [3]:
# Expected dimensions
EXPECTED_TIMESTEPS = 128
EXPECTED_FEATURES = 9
EXPECTED_CLASSES = 6

assert X_train.shape[1:] == (EXPECTED_TIMESTEPS, EXPECTED_FEATURES)
assert X_val.shape[1:] == (EXPECTED_TIMESTEPS, EXPECTED_FEATURES)
assert X_test.shape[1:] == (EXPECTED_TIMESTEPS, EXPECTED_FEATURES)

assert len(np.unique(y_train)) == EXPECTED_CLASSES
assert len(np.unique(y_val)) == EXPECTED_CLASSES
assert len(np.unique(y_test)) == EXPECTED_CLASSES

print("Input shape:", X_train.shape[1:])
print("Number of classes:", EXPECTED_CLASSES)
print("Training samples:", X_train.shape[0])
print("Validation samples:", X_val.shape[0])
print("Test samples:", X_test.shape[0])

print("\nData structure verification: PASSED")

Input shape: (128, 9)
Number of classes: 6
Training samples: 5551
Validation samples: 1801
Test samples: 2947

Data structure verification: PASSED


## 4. Activity Labels

The dataset contains six human activity classes.

The integer labels are mapped to their corresponding activity names using the
metadata stored in the processed dataset.

In [4]:
print("Activity label mapping:\n")

for label, activity in enumerate(activity_names):
    print(f"{label}: {activity}")

print("\nTraining class distribution:\n")

train_class_counts = pd.Series(y_train).value_counts().sort_index()

for label, count in train_class_counts.items():
    print(f"{label} - {activity_names[label]}: {count}")

Activity label mapping:

0: Walking
1: Walking Upstairs
2: Walking Downstairs
3: Sitting
4: Standing
5: Laying

Training class distribution:

0 - Walking: 888
1 - Walking Upstairs: 797
2 - Walking Downstairs: 744
3 - Sitting: 993
4 - Standing: 1053
5 - Laying: 1076


## 5. Experimental Configuration

The candidate CNN architectures will be compared under controlled training
conditions.

The main experimental variable will be the CNN architecture/capacity.

The following conditions will remain consistent across candidates:

- Same training data
- Same validation data
- Same preprocessing
- Same loss function
- Same optimizer configuration
- Same batch size
- Same maximum number of epochs
- Same callback strategy
- Same random seed

The test set will not be used during candidate selection.

Exact CNN architecture parameters such as filter counts, kernel sizes and
number of convolutional blocks will be defined in the model-design phase.

In [5]:
# Experiment-level configuration
SEED = 42

# Data dimensions
INPUT_SHAPE = X_train.shape[1:]
NUM_CLASSES = len(activity_names)

# Candidate selection
PRIMARY_SELECTION_METRIC = "val_loss"

# Test set policy
TEST_USED_DURING_SELECTION = False

print("Experiment configuration")
print("------------------------")
print("Random seed:", SEED)
print("Input shape:", INPUT_SHAPE)
print("Number of classes:", NUM_CLASSES)
print("Primary selection metric:", PRIMARY_SELECTION_METRIC)
print("Test used during candidate selection:", TEST_USED_DURING_SELECTION)

Experiment configuration
------------------------
Random seed: 42
Input shape: (128, 9)
Number of classes: 6
Primary selection metric: val_loss
Test used during candidate selection: False


## 6. Candidate Evaluation Records

For each CNN candidate, the experiment will record both predictive and
computational characteristics.

### Predictive performance
- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

### Training behaviour
- Training loss
- Validation loss
- Training accuracy
- Validation accuracy
- Number of epochs completed

### Model complexity
- Total parameters
- Trainable parameters

### Computational efficiency
- Training time
- Inference time

These measurements will support candidate selection and the later analysis of
generalization, model complexity, computational cost, convergence and errors.

In [6]:
candidate_results = []

print("Candidate result structure initialized.")
print("Results will be added after the CNN candidates are implemented.")

Candidate result structure initialized.
Results will be added after the CNN candidates are implemented.


## 7. Model Selection and Final Evaluation Protocol

The CNN experiment follows a two-stage evaluation process.

### Stage 1 — Candidate development and selection

Each candidate CNN is trained using the training set.

The validation set is used to:

- monitor training behaviour
- compare candidate architectures
- identify overfitting or underfitting
- select the final CNN architecture

The test set is not used during this stage.

### Stage 2 — Final evaluation

After the final CNN architecture and training configuration have been selected,
the test set is used for the final evaluation.

The final test results are reported separately from the validation results.

This prevents the test data from influencing model selection.

In [7]:
# Explicit experiment split policy
EXPERIMENT_SPLITS = {
    "training": "Used for model fitting",
    "validation": "Used for candidate comparison and model selection",
    "test": "Reserved for final evaluation only"
}

for split, purpose in EXPERIMENT_SPLITS.items():
    print(f"{split.capitalize():12} -> {purpose}")

Training     -> Used for model fitting
Validation   -> Used for candidate comparison and model selection
Test         -> Reserved for final evaluation only
